# 🚶 行人目标检测 — YOLOv8 训练

1. 运行时 → 更改运行时类型 → **T4 GPU**
2. 逐个单元格 Ctrl+Enter 执行

## 第1步：安装依赖

In [ ]:
!pip install ultralytics roboflow 2>&1 | tail -3

## 第2步：挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 第3步：从 Roboflow 下载数据集

In [ ]:
import os
if not os.path.exists('/content/drive/MyDrive/datasets/pedestrian'):
    from roboflow import Roboflow
    rf = Roboflow(api_key="UnmHkLh06PVV5rkrlUjl")
    project = rf.workspace("chris-kydks").project("people-detection-2csbw")
    dataset = project.version(11).download("yolov8")
    print(f"下载完成: {dataset.location}")
    import shutil
    shutil.move(dataset.location, '/content/drive/MyDrive/datasets/pedestrian')
else:
    print("已有数据集，跳过下载")

!mkdir -p /content/datasets/
!cp -r /content/drive/MyDrive/datasets/pedestrian /content/datasets/pedestrian
print("✅ 数据已复制到 Colab 本地磁盘")

## 第4步：创建配置文件

In [ ]:
yaml_content = """
path: /content/datasets/pedestrian
train: train/images
val: valid/images
nc: 1
names: ['person']
"""
with open('/content/pedestrian.yaml', 'w') as f:
    f.write(yaml_content)
!cat /content/pedestrian.yaml

## 第5步：开始训练 🚀

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

model.train(
    data='/content/pedestrian.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='pedestrian-det',
    patience=10,
    lr0=0.001,
    augment=True,
    device=0,
)

## 第6步：评估效果

In [ ]:
metrics = model.val()
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")

## 第7步：训练曲线

In [ ]:
from IPython.display import Image, display

# 找到最新的训练结果目录
import glob
result_dirs = sorted(glob.glob('runs/detect/pedestrian-det*/results.png'))
if result_dirs:
    print(f"找到: {result_dirs[-1]}")
    display(Image(filename=result_dirs[-1]))
else:
    print("没找到 results.png，检查 runs/detect/ 目录")

## 第8步：测试一张图

In [ ]:
model.predict(source='/content/datasets/pedestrian/valid/images/', conf=0.35, save=True)
import glob
imgs = glob.glob('runs/detect/predict/*.jpg')
if imgs:
    display(Image(filename=imgs[0]))

## 第9步：保存模型到 Google Drive

In [ ]:
import shutil
import glob

# 自动找最新的 best.pt
best_files = sorted(glob.glob('runs/detect/pedestrian-det*/weights/best.pt'))
if best_files:
    best_pt = best_files[-1]
    print(f"找到: {best_pt}")
    shutil.copy(best_pt, '/content/drive/MyDrive/pedestrian_best.pt')
    print("✅ 模型已保存到 Google Drive")
    print("下载到本地后放到 models/ 目录下，设 MODEL_NAME=pedestrian_best.pt 启动 app.py")
else:
    print("⚠️ 没找到 best.pt，检查 runs/detect/")